# OLMo 3.1 32B Subset Planner With Clean uv Subprocess

Purpose: avoid Colab kernel Torch/vLLM contamination by creating a clean uv virtual environment and running the benchmark as standalone Python subprocesses.

This notebook does not import `torch` or `vllm` in the notebook kernel. Each checkpoint run is a separate process using `.venv-colab-vllm/bin/python`.


## 1. Clone Or Update Repo

Edit `REPO_URL` if the GitHub remote is different or private. If private, authenticate with GitHub before running this cell.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/ndalton12/carry-trace.git'  # Change if needed.
BRANCH = 'main'
REPO_DIR = Path('/content/carry-trace')

if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print(f'Repo: {REPO_DIR}')
print(f'Commit: {commit}')
print(f'Notebook Python: {sys.version}')


## 2. Create Clean uv Environment

This intentionally installs a pinned Torch/vLLM pair into `.venv-colab-vllm`. If this cell succeeds, no runtime restart should be needed because the notebook kernel never imports those packages.


In [ ]:
!python -m pip install -U uv
!uv venv .venv-colab-vllm --python 3.12 --clear
!uv pip install --python .venv-colab-vllm/bin/python -e . --no-deps
!uv pip install --python .venv-colab-vllm/bin/python 'torch==2.8.0' 'torchvision==0.23.0' 'torchaudio==2.8.0' --index-url https://download.pytorch.org/whl/cu128
!uv pip install --python .venv-colab-vllm/bin/python 'vllm==0.10.2'
!uv pip install --python .venv-colab-vllm/bin/python 'accelerate>=1.0.0' 'datasets>=3.0.0' 'matplotlib>=3.9.0' 'numpy>=2.0.0' 'pandas>=2.2.0' 'pyarrow>=17.0.0' 'pydantic>=2.9.0' 'pyyaml>=6.0.2' 'rich>=13.8.0' 'seaborn>=0.13.2' 'transformers>=4.55.2' 'typer>=0.12.5' 'huggingface_hub[hf_transfer]>=0.23.0'
!./.venv-colab-vllm/bin/python -c "import torch, vllm; print('venv python ok'); print('torch', torch.__version__, 'cuda', torch.version.cuda, 'cuda_available', torch.cuda.is_available()); print('vllm', vllm.__version__)"
!nvidia-smi


## 3. Configure Benchmark

The default candidate pool is 18 examples: `3 digit lengths x 3 slices x 2 prompt modes`. Instruct uses a 2048-token cap. Think uses 4096 thinking tokens plus 100 reserved final-answer tokens.


In [ ]:
from pathlib import Path
import subprocess
import time

VENV_PY = REPO_DIR / '.venv-colab-vllm' / 'bin' / 'python'
SCRIPT = REPO_DIR / 'scripts' / 'olmo31_subset_planner.py'

BENCH_NAME = 'colab_olmo31_32b_think_instruct_subset_planner'
RUN_ROOT = REPO_DIR / 'runs' / 'subset_planner' / f'{BENCH_NAME}-{time.strftime("%Y%m%d-%H%M%S")}'

RUN_INSTRUCT = True
RUN_THINK = True

COMMON_ARGS = [
    '--name', BENCH_NAME,
    '--seed', '20260611',
    '--run-root', str(RUN_ROOT),
    '--output-dir', str(REPO_DIR / 'data' / 'generated'),
    '--examples-per-slice-per-length', '1',
    '--digit-lengths', '4,8,12',
    '--slices', 'no_carry,isolated_carry,long_carry_chain',
    '--prompt-modes', 'answer_only,free_cot',
    '--batch-size', '2',
    '--dtype', 'bfloat16',
    '--gpu-memory-utilization', '0.90',
    '--enforce-eager',
    '--target-hours-per-checkpoint', '24',
    '--temperature', '0.6',
    '--top-p', '0.95',
    '--instruct-max-new-tokens', '2048',
    '--think-thinking-tokens', '4096',
    '--thinking-final-answer-tokens', '100',
]

print('venv python:', VENV_PY)
print('script:', SCRIPT)
print('run root:', RUN_ROOT)


## 4. Run 32B Instruct

Progress is printed by the subprocess after each completed example record. Since `batch_size=2`, progress updates arrive after each vLLM batch completes.


In [ ]:
if RUN_INSTRUCT:
    subprocess.run([str(VENV_PY), str(SCRIPT), '--checkpoint', 'instruct', *COMMON_ARGS], check=True)
else:
    print('Skipping Instruct run.')


## 5. Run 32B Think

This is a separate process from Instruct, so the previous vLLM engine and CUDA allocations are gone when this starts.


In [ ]:
if RUN_THINK:
    subprocess.run([str(VENV_PY), str(SCRIPT), '--checkpoint', 'think', *COMMON_ARGS], check=True)
else:
    print('Skipping Think run.')


## 6. Summarize Viable Subset

This reads saved artifacts and prints runtime, accuracy, force-close rate, cell-level summaries, and the recommended paired examples-per-cell for a 24-hour budget.


In [ ]:
subprocess.run([str(VENV_PY), str(SCRIPT), '--checkpoint', 'summary', *COMMON_ARGS], check=True)


## 7. Inspect Saved Outputs

Use this lightweight notebook-kernel cell to inspect artifact paths. It does not import Torch or vLLM.


In [ ]:
print('Artifacts:')
for path in sorted(RUN_ROOT.glob('*')):
    print(path)

print('\nRecommendation JSON:')
print((RUN_ROOT / 'subset_recommendation.json').read_text() if (RUN_ROOT / 'subset_recommendation.json').exists() else 'not written yet')
